In [21]:
##### IMPORTS

# General
import re
import numpy as np

from pathlib import Path
from typing import Optional, Tuple, List

# Other
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.io import output_notebook
output_notebook()

Loading BokehJS ...

In [22]:
##### CONFIGURATIONS

# Directories
outputs = Path(r"C:\Users\Alex\Desktop\Picaso\NN_project\cloudy_spectra_code\outputs")
# output_path = 'home/al864695/ouputs'
out_sin = r"C:\Users\Alex\Desktop\Picaso\NN_project\cloudy_spectra_code\outputs\t450.0g3.5f1.0k1e10c2345678.npz"

# Dictionary for cloud naming conventions
cloud_dict  = {'Fe': '1', 'H2O': '2', 'KCl': '3', 'Mg2SiO4': '4',
               'MgSiO3': '5', 'MnS': '6', 'NH3': '7', 'Na2S': '8'}
cloud_map   = {v: k for k, v in cloud_dict.items()}

In [23]:
##### PARSE FILE NAMES

# Filenames like: t1450.0g4.5f2.0k1e9c12358.npz
fname = re.compile(r"""^t(?P<Teff>\d+(?:\.\d+)?)
                   g(?P<logg>-?\d+(?:\.\d+)?)
                   f(?P<fsed>-?\d+(?:\.\d+)?)
                   (?P<ktag>k[0-9eE+\-\.]+)
                   (?:c(?P<clouds>\d+))?
                   \.npz$""", re.VERBOSE)

def parsekzz(ktag: str) -> float:
    s = ktag[1:]
    if re.fullmatch(r"\d+", s):
        return 10.0 ** int(s)
    try:
        return float(s)
    except Exception:
        return float('nan')

def _decode_clouds(code: Optional[str]) -> Tuple[str, List[str]]:
    if not code:
        return "", []
    names = []
    for d in code:
        name = cloud_map.get(d)
        if name:
            names.append(name)
    return code, names

def params_from_filename(p: Path):
    m = fname.match(p.name)
    if not m:
        raise ValueError(f"Filename does not match expected pattern: {p.name}")
    Teff = float(m.group("Teff"))
    logg = float(m.group("logg"))
    fsed = float(m.group("fsed"))
    kzz  = parsekzz(m.group("ktag"))
    clouds_code, clouds = _decode_clouds(m.group("clouds"))
    return dict(Teff=Teff, logg=logg, fsed=fsed, kzz=kzz,
                clouds_code=clouds_code, clouds=clouds, filename=p.name)

In [28]:
##### OPEN AND READ .NPZ FILES

# Read .NPZ
def readnpz(npz_path):
    """
    Return (w_um, F, params dict) from a MARGE per-case .npz:
      - x: wavelength [micron], saved with shape (1, N)
      - y: flux per wavenumber [erg cm^-2 s^-1 (cm^-1)^-1], shape (1, N)
    """
    npz_path = Path(npz_path)
    d = np.load(npz_path, allow_pickle=False)
    x = np.asarray(d["x"])
    y = np.asarray(d["y"])
    if x.ndim == 2 and x.shape[0] == 1:
        x = x[0]
    if y.ndim == 2 and y.shape[0] == 1:
        y = y[0]
    W = x.astype(float)
    F = y.astype(float)

    params = params_from_filename(npz_path)
    return W, F, params

# STATS (optional)
def stats(npz_path):
    W, F, p = readnpz(npz_path)

    cloud_str = (f"c{p['clouds_code']}  "
                 f"({', '.join(p['clouds'])})") if p['clouds_code'] else "(none)"

    print(f"File: {p['filename']}")
    print(f"Teff [K]: {p['Teff']:.0f}")
    print(f"logg [cgs]: {p['logg']:.2f}")
    print(f"f_sed: {p['fsed']:.2f}")
    print(f"kzz [cm^2 s^-1]: {p['kzz']:.3e}")
    print(f"Clouds: {cloud_str}")
    print(f"Wave points: {W.size}")
    print(f"Wave range (micron): {W.min():.4f} - {W.max():.4f}")
    print(f"F min/max: {np.nanmin(F):.3e} / {np.nanmax(F):.3e}")

    return dict(wavelength_um=W, F=F, **p)

# Plot
def plot_npz(npz_path):
    W, F, p = readnpz(npz_path)

    clouds_label = (f", clouds=({', '.join(p['clouds'])})"
                    if p['clouds_code'] else ", clouds=none")

    title = (f"Teff={p['Teff']:.0f}, logg={p['logg']:.2f}, "
             f"f_sed={p['fsed']:.2f}, kzz={p['kzz']:.2e}"
             f"{clouds_label}")

    src = ColumnDataSource(dict(wavelength_um=W, F=F))
    tools = "pan,wheel_zoom,box_zoom,reset,save"
    fig = figure(title=title,
                 x_axis_label="Wavelength (micron)",
                 y_axis_label="F (erg cm^-2 s^-1 cm^-1)",
                 sizing_mode="stretch_width", height=420,
                 tools=tools, output_backend="canvas")

    fig.add_tools(HoverTool(tooltips=[("λ (μm)", "@wavelength_um{0.000}"),
                                      ("F", "@F{0.00e}")],
                            mode="vline"))
    fig.line('wavelength_um', 'F', source=src, line_width=2)
    show(fig)

In [29]:
stats(out_sin)
plot_npz(out_sin)

File: t450.0g3.5f1.0k1e10c2345678.npz
Teff [K]: 450
logg [cgs]: 3.50
f_sed: 1.00
kzz [cm^2 s^-1]: 1.000e+10
Clouds: c2345678  (H2O, KCl, Mg2SiO4, MgSiO3, MnS, NH3, Na2S)
Wave points: 844
Wave range (micron): 0.3005 - 4.9914
F min/max: 2.036e-06 / 1.977e+10
